In [ ]:
print("hello")

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

In [21]:
df = pd.read_csv("project_1_python.csv")

df

,index,iso_code,continent,location,date,total_cases,new_cases,total_deaths,new_deaths,hosp_patients,...,people_vaccinated,people_fully_vaccinated,total_boosters,new_vaccinations,population,median_age,gdp_per_capita,life_expectancy,latitude,longitude
0,0,AFG,Asia,Afghanistan,2020-02-24,5.0,5.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,39835428.0,18.6,1803.987,64.83,33.0,65.0
1,1,AFG,Asia,Afghanistan,2020-02-25,5.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,39835428.0,18.6,1803.987,64.83,33.0,65.0
2,2,AFG,Asia,Afghanistan,2020-02-26,5.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,39835428.0,18.6,1803.987,64.83,33.0,65.0
3,3,AFG,Asia,Afghanistan,2020-02-27,5.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,39835428.0,18.6,1803.987,64.83,33.0,65.0
4,4,AFG,Asia,Afghanistan,2020-02-28,5.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,39835428.0,18.6,1803.987,64.83,33.0,65.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180472,190608,ZWE,Africa,Zimbabwe,2022-06-14,254155.0,0.0,5521.0,0.0,NaN,...,6270096.0,4563366.0,1026048.0,NaN,15092171.0,19.6,1899.775,61.49,-20.0,30.0
180473,190609,ZWE,Africa,Zimbabwe,2022-06-15,254387.0,232.0,5525.0,4.0,NaN,...,6271703.0,4567466.0,1027822.0,7481.0,15092171.0,19.6,1899.775,61.49,-20.0,30.0
180474,190610,ZWE,Africa,Zimbabwe,2022-06-16,254502.0,115.0,5526.0,1.0,NaN,...,6274305.0,4570349.0,1029463.0,7126.0,15092171.0,19.6,1899.775,61.49,-20.0,30.0
180475,190611,ZWE,Africa,Zimbabwe,2022-06-17,254753.0,251.0,5533.0,7.0,NaN,...,6276402.0,4574222.0,1031790.0,8297.0,15092171.0,19.6,1899.775,61.49,-20.0,30.0


In [ ]:
df.info()

In [22]:
df = df.rename(columns={"location":"country"})
df_countries = df[["iso_code","continent","country","population","life_expectancy"]].drop_duplicates()
df_countries = df_countries.sort_values(by="population",ascending=False)

ten_top_pop_countries = df_countries.head(10)

ten_top_pop_countries


,iso_code,continent,country,population,life_expectancy
33499,CHN,Asia,China,1.444216e+09,76.91
74570,IND,Asia,India,1.393409e+09,69.66
170959,USA,North America,United States,3.329151e+08,78.86
75441,IDN,Asia,Indonesia,2.763618e+08,71.72
122998,PAK,Asia,Pakistan,2.251999e+08,67.27
21805,BRA,South America,Brazil,2.139934e+08,75.88
118134,NGA,Africa,Nigeria,2.114007e+08,54.69
12663,BGD,Asia,Bangladesh,1.663035e+08,72.59
133450,RUS,Europe,Russia,1.459120e+08,72.58
103850,MEX,North America,Mexico,1.302622e+08,75.05


In [ ]:
graph = sns.catplot(
    data=ten_top_pop_countries,
    x=ten_top_pop_countries["population"]/1e6,
    y=ten_top_pop_countries["country"],
    kind="bar")

sns.set_style("whitegrid")
plt.xlabel("population(millions)")
graph.set_xticklabels(rotation=45, ha="right")

In [ ]:
fig = px.scatter(
    data_frame=df_countries,
    x="population",
    y="life_expectancy",
    title="Population vs life expectancy",
    log_x=True,
    color="continent",
    hover_name="country",
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.show()

In [ ]:
df_cz_de = df[df["country"].isin(["Czechia","Germany"])]

df_cz_de

In [ ]:
fig = px.line(
    data_frame=df_cz_de,
    x="date",
    y="new_cases",
    color="country",
    color_discrete_sequence=px.colors.qualitative.Antique,
    title="Covid cases in Czechia vs Germany over time"
)

fig.show()

In [ ]:
# now I am going to rescale it to population of both countries:

df_cz_de["new_cases_per_1000_people"] = df_cz_de["new_cases"]/df_cz_de["population"]/1000

fig = px.line(
    data_frame=df_cz_de,
    x="date",
    y="new_cases_per_1000_people",
    color="country",
    color_discrete_sequence=px.colors.qualitative.Antique,
    title="Covid cases in Czechia vs Germany per thousand people over time"
)

# this line is from Claude, it fixes y axis values to something not containing Greek letters:
fig.update_yaxes(exponentformat="none") 

fig.show()

In [ ]:
df = df.dropna(subset=["total_cases"])
df_countries = df[["iso_code","continent","country","population","life_expectancy","total_cases"]].drop_duplicates()
df_countries = df_countries.sort_values(by="population",ascending=False)

ten_top_pop_countries = df_countries.head(10)

ten_top_pop_countries


In [ ]:
#df["cases_to_population"] = df["total_cases"]/df["population"]

"""
I am not sure how to understand
the task "2. Make the size of the marker dependent on the ratio of cases to population in the country."
There are multiple values of total cases for each country.
Anyway, I've decided to pick the highest value, presumably this is total cases for all time 
(to publication of dataset, anyway).
"""
df_countries2 = df.sort_values("total_cases", ascending=False).drop_duplicates(subset="country", keep="first")

df_countries2["cases_to_population"] = df_countries2["total_cases"]/df_countries2["population"]

fig = px.scatter_geo(
    data_frame=df_countries2,
    locations="iso_code",
    locationmode="ISO-3",      
    color="continent",
    size="cases_to_population",             
    hover_name="country",       
    hover_data="total_cases",
    title="Cases to population"
)

"""
There is a task to set the map to a dark theme.
Imho It doesn't look good, unfortunately, but anyway:
"""
fig.update_layout(template="plotly_dark",height=800)

fig.show()


In [25]:
app = dash.Dash()

font_styles={"fontFamily": "verdana", "color": "#444"}

app.layout = html.Div(
    [html.H3(id="title", style=font_styles),
     
dcc.Dropdown(
        id="country-picker",
        clearable=False,
        # original df lists country as "location", I've renamed column somewhere above to "country",
        # so this only works if relevant cells were run before:
        options=[{"label": country, 'value': country} 
                 for country in df["country"].unique()],
        value = "Czechia",
        style=font_styles
        ),
html.Div([
dcc.Graph(id="graph_cases", style={'width': '50%'}),
dcc.Graph(id="graph_deaths", style={'width': '50%'})
], style = {"display":"flex"})

])

@app.callback(
    Output("title", "children"),
    Output("graph_cases", "figure"),
    Output("graph_deaths", "figure"),
    Input("country-picker","value")
)
def update_dashboard(country):
    title = f"Cumulative number of positive cases and deaths in {country}"
    selected_country = df[df["country"] == country]
    cases = px.line(selected_country, x="date", y="total_cases")
    deaths = px.line(selected_country, x="date", y="total_deaths")
    return title, cases, deaths

if __name__ == "__main__":
    app.run(debug=True, host="127.0.0.1", port=8050)